In [1]:
import pandas as pd
pd.__version__


'3.0.0'

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta
import random

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)
random.seed(42)

n = 50

# --- USERS RAW CSV ---
users_raw_df = pd.DataFrame({
    "user_id": [f"U{1000+i}" for i in range(n)] + [None],   # missing user_id
    "status": np.random.choice(["Active", "ACTIVE ", " inactive", "Cancelled"], n+1),
    "signup_date": [
        (datetime.today() - timedelta(days=random.randint(30, 900))).strftime("%Y-%m-%d")
        for _ in range(n+1)
    ],
    "monthly_fee": np.random.choice([9.99, 14.99, 19.99, None], n+1),
    "months_active": np.random.randint(1, 24, n+1),
    "age": np.random.choice([18, 25, 30, "unknown", None], n+1),
    "country_code": np.random.choice(["AU", "US", "IN", "GB", None], n+1)
})

users_raw_df.to_csv(RAW_DIR / "users_raw.csv", index=False)

# --- SUPPORT RAW JSON ---
support = []
valid_ids = users_raw_df["user_id"].dropna().astype(str).tolist()

for i in range(80):
    support.append({
        "ticket_id": f"T{i+1}",
        "user_id": random.choice(valid_ids),
        "issue_type": random.choice(["billing", "technical", "account"]),
        "created_at": (datetime.today() - timedelta(days=random.randint(1, 365))).isoformat()
    })

with open(RAW_DIR / "support_raw.json", "w", encoding="utf-8") as f:
    json.dump(support, f, indent=2)

"Raw CSV and JSON created"


'Raw CSV and JSON created'

In [3]:
from pathlib import Path
list(Path("../data/raw").glob("*"))


[WindowsPath('../data/raw/support_raw.json'),
 WindowsPath('../data/raw/users_raw.csv')]

In [4]:
## 3. Cleaning & Validation


In [5]:
import pandas as pd
import json

users = pd.read_csv("../data/raw/users_raw.csv")

with open("../data/raw/support_raw.json", "r", encoding="utf-8") as f:
    support_json = json.load(f)
support = pd.json_normalize(support_json)

users.head(), support.head()


(  user_id     status signup_date  monthly_fee  months_active  age country_code
 0   U1000   inactive  2024-03-27        14.99             12   30           IN
 1   U1001  Cancelled  2025-09-18        19.99              2   30           IN
 2   U1002     Active  2025-12-16          NaN             10   18           AU
 3   U1003   inactive  2023-12-13        19.99              4   30          NaN
 4   U1004   inactive  2025-04-04          NaN             14  NaN           AU,
   ticket_id user_id issue_type                  created_at
 0        T1   U1006  technical  2025-08-16T20:46:58.576100
 1        T2   U1038  technical  2026-01-17T20:46:58.576100
 2        T3   U1046  technical  2025-05-10T20:46:58.576100
 3        T4   U1007  technical  2025-12-30T20:46:58.576100
 4        T5   U1035  technical  2025-03-24T20:46:58.576100)

In [6]:
users["status"] = users["status"].astype(str).str.strip().str.lower()

users["status"].value_counts(dropna=False)


status
active       21
cancelled    16
inactive     14
Name: count, dtype: int64

In [7]:
users["signup_date"] = pd.to_datetime(users["signup_date"], errors="coerce")

users.dtypes


user_id                     str
status                      str
signup_date      datetime64[us]
monthly_fee             float64
months_active             int64
age                         str
country_code                str
dtype: object

In [8]:
import numpy as np

users["monthly_fee"] = pd.to_numeric(users["monthly_fee"], errors="coerce")

median_fee = users["monthly_fee"].median()
users["monthly_fee"] = users["monthly_fee"].fillna(median_fee)

median_fee, users["monthly_fee"].isna().sum()


(np.float64(14.99), np.int64(0))

In [9]:
users["age"] = pd.to_numeric(users["age"], errors="coerce")

users["age"].dtype, users["age"].isna().sum()


(dtype('float64'), np.int64(23))

In [10]:
# Count issues before dropping
missing_user_id = users["user_id"].isna().sum()
dup_user_id = users["user_id"].duplicated().sum()

missing_user_id, dup_user_id


(np.int64(1), np.int64(0))

In [11]:
users = users.dropna(subset=["user_id"])

# Verify
users["user_id"].isna().sum(), users["user_id"].duplicated().sum(), users.shape


(np.int64(0), np.int64(0), (50, 7))

In [12]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

clean_path = PROCESSED_DIR / "users_cleaned.csv"
users.to_csv(clean_path, index=False)

str(clean_path)


'..\\data\\processed\\users_cleaned.csv'

In [13]:
## 4. Feature Engineering


In [14]:
users["is_active_subscription"] = (users["status"] == "active").astype(int)
users["is_active_subscription"].value_counts()


is_active_subscription
0    29
1    21
Name: count, dtype: int64

In [15]:
users["months_active"] = pd.to_numeric(users["months_active"], errors="coerce").fillna(0)

users["revenue_estimate"] = users["monthly_fee"] * users["months_active"]

users[["monthly_fee", "months_active", "revenue_estimate"]].head()


,monthly_fee,months_active,revenue_estimate
0,14.99,12,179.88
1,19.99,2,39.98
2,14.99,10,149.90
3,19.99,4,79.96
4,14.99,14,209.86


In [17]:
users["subscription_length_days"] = users["months_active"] * 30

users[["months_active", "subscription_length_days"]].head()


,months_active,subscription_length_days
0,12,360
1,2,60
2,10,300
3,4,120
4,14,420


In [18]:
from pathlib import Path

FINAL_DIR = Path("../data/processed")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

final_path = FINAL_DIR / "users_analytics_ready.csv"
users.to_csv(final_path, index=False)

str(final_path)


'..\\data\\processed\\users_analytics_ready.csv'

In [19]:
summary = {
    "total_users": len(users),
    "active_users": int(users["is_active_subscription"].sum()),
    "inactive_users": int((users["is_active_subscription"] == 0).sum()),
    "total_estimated_revenue": round(users["revenue_estimate"].sum(), 2),
    "avg_revenue_per_user": round(users["revenue_estimate"].mean(), 2),
}

summary


{'total_users': 50,
 'active_users': 21,
 'inactive_users': 29,
 'total_estimated_revenue': np.float64(7944.68),
 'avg_revenue_per_user': np.float64(158.89)}

In [20]:
import requests
import time

def fetch_country_name(code, retries=3, backoff=1):
    """
    Fetch country name from REST Countries API.
    Returns None if API fails.
    """
    if pd.isna(code):
        return None

    code = str(code).strip().upper()
    url = f"https://restcountries.com/v3.1/alpha/{code}"

    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=5)

            if response.status_code == 200:
                data = response.json()
                if isinstance(data, list) and len(data) > 0:
                    return data[0]["name"]["common"]
                return None

            # Retry on server / rate-limit issues
            if response.status_code in [429, 500, 502, 503, 504]:
                time.sleep(backoff)

        except requests.RequestException:
            time.sleep(backoff)

    return None


In [21]:
# Get unique country codes
country_codes = users["country_code"].dropna().unique()
country_codes


<StringArray>
['IN', 'AU', 'US', 'GB']
Length: 4, dtype: str

In [22]:
country_lookup = {}

for code in country_codes:
    country_lookup[code] = fetch_country_name(code)

country_lookup


{'IN': 'India',
 'AU': 'Australia',
 'US': 'United States',
 'GB': 'United Kingdom'}

In [23]:
users["country_name"] = users["country_code"].map(country_lookup)

users[["country_code", "country_name"]].drop_duplicates()


,country_code,country_name
0,IN,India
2,AU,Australia
3,NaN,NaN
6,US,United States
7,GB,United Kingdom


In [24]:
from pathlib import Path

ENRICHED_PATH = Path("../data/processed/users_enriched.csv")
users.to_csv(ENRICHED_PATH, index=False)

str(ENRICHED_PATH)


'..\\data\\processed\\users_enriched.csv'